# Phase 1: Data Infrastructure & Empirical Diagnostics  
## Notebook 01_07 — Futures Calendar and Roll Yield Decomposition

### Research question

How can we transform individual futures contracts into an auditable calendar, curve-feature, and return-attribution dataset that separates price movement, roll effects, collateral return, and term-structure information?

This notebook builds on `01_03_continuous_futures_contract_rolling`.

Instead of focusing only on how to create a smooth continuous chart, this notebook asks a deeper question:

> What economic information is contained in the futures calendar and term structure, and how can it be decomposed without confusing adjusted chart history with tradable returns?

The notebook covers:

1. synthetic futures calendar generation;
2. term-structure simulation with contango/backwardation regimes;
3. contract metadata and time-to-expiry;
4. front/second/third contract curve features;
5. calendar roll schedule construction;
6. held-contract excess return;
7. roll-gap proxy return;
8. collateral return;
9. total collateralised futures return;
10. spot proxy return;
11. roll-yield decomposition diagnostics;
12. carry, slope, and curvature signals;
13. regime analysis by contango/backwardation state;
14. saved roll calendar, curve features, and return attribution tables.

The goal is to build a futures data layer that can support CTA research, carry strategies, curve modelling, and execution-aware rolling.

## 1. Motivation

A continuous futures chart is useful, but it hides important structure.

A futures strategy needs to know:

- which contract was held;
- when the contract was rolled;
- what the curve looked like at the roll;
- whether the market was in contango or backwardation;
- how much return came from price movement versus curve roll;
- what collateral return would have been earned on unused cash.

This matters because futures returns are not the same as spot returns.

A fully collateralised futures position is often decomposed conceptually into:

$$
\begin{aligned}
\text{Total return} &\approx \text{Futures excess return} \\
&\quad + \text{Collateral return}
\end{aligned}
$$
For commodity futures, the futures excess return is often analysed using:

$$
\begin{aligned}
\text{Futures excess return} &\approx \text{Spot proxy return} \\
&\quad + \text{Roll yield proxy}
\end{aligned}
$$
This is an attribution framework. It is useful, but it must be handled carefully because “roll yield” can be defined differently across papers, index providers, and practitioners.

## 2. Contango and backwardation

Let $F(t,T_1)$ be the near contract price and $F(t,T_2)$ be the next contract price, with $T_2 > T_1$.

If:

$$
F(t,T_2) > F(t,T_1)
$$
the front of the curve is in **contango**.

If:

$$
F(t,T_2) < F(t,T_1)
$$
the front of the curve is in **backwardation**.

A simple annualised front-next carry signal is:

$$
\begin{aligned}
\text{carry}_t &= \frac{\log F(t,T_1) - \log F(t,T_2)} {\tau_2 - \tau_1}
\end{aligned}
$$
where:

$$
\tau_i = \frac{T_i - t}{365}
$$
This signal is positive when the front contract is above the next contract, which corresponds to backwardation at the front of the curve. It is negative in contango.

This convention is useful for a long futures carry signal: positive carry is generally favourable for long exposure, while negative carry is generally costly.

## 3. Imports and configuration

The notebook is self-contained.

It uses synthetic futures data so that the calendar and decomposition can be audited without external data.

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

In [ ]:
@dataclass(frozen=True)
class FuturesCalendarConfig:
    """
    Configuration for the synthetic futures calendar.
    """
    root: str = "SYN"
    start: str = "2020-01-01"
    end: str = "2024-12-31"
    max_days_to_expiry: int = 240
    roll_days_before_expiry: int = 7
    contract_multiplier: float = 100.0
    annual_collateral_rate: float = 0.035
    seed: int = 42


config = FuturesCalendarConfig()
config

## 4. Futures month codes and expiry calendar

Futures contracts usually have a root symbol, month code, and year code.

For example:

```text
SYNH24
```

can be read as:

- `SYN`: root symbol;
- `H`: March contract month;
- `24`: year 2024.

This notebook uses monthly expiries based on the third Friday of each month.

In [ ]:
MONTH_CODES = {
    1: "F",
    2: "G",
    3: "H",
    4: "J",
    5: "K",
    6: "M",
    7: "N",
    8: "Q",
    9: "U",
    10: "V",
    11: "X",
    12: "Z",
}


def contract_code(root: str, expiry: pd.Timestamp) -> str:
    """
    Create a futures contract code from root and expiry month.
    """
    month_code = MONTH_CODES[int(expiry.month)]
    year_code = str(expiry.year)[-2:]
    return f"{root}{month_code}{year_code}"


def third_friday(year: int, month: int) -> pd.Timestamp:
    """
    Return the third Friday of a given month, UTC-normalised.
    """
    dates = pd.date_range(
        start=f"{year}-{month:02d}-01",
        end=f"{year}-{month:02d}-28",
        freq="D",
        tz="UTC"
    )
    fridays = dates[dates.weekday == 4]
    return fridays[2]


def make_monthly_expiries(start_year: int, end_year: int) -> pd.DataFrame:
    """
    Build a contract metadata table with monthly expiries.
    """
    rows = []

    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            expiry = third_friday(year, month)
            rows.append({
                "root": config.root,
                "contract": contract_code(config.root, expiry),
                "delivery_year": year,
                "delivery_month": month,
                "expiry": expiry,
                "contract_multiplier": config.contract_multiplier,
            })

    return pd.DataFrame(rows)


contract_metadata = make_monthly_expiries(
    start_year=pd.Timestamp(config.start).year,
    end_year=pd.Timestamp(config.end).year + 1
)

contract_metadata.head()

## 5. Simulating a futures term structure

We simulate a latent spot-like index and a futures curve.

The synthetic futures price is generated as:

$$
\begin{aligned}
F(t,T) &= S_t \exp\Big( c_t \tau \\
&\quad + q_t \tau^2 \\
&\quad + s(t,T) \\
&\quad + \epsilon_{t,T} \Big)
\end{aligned}
$$
where:

- $S_t$ is a latent spot proxy;
- $c_t$ is a time-varying carry/slope factor;
- $q_t$ is a curvature factor;
- $s(t,T)$ is a seasonal component;
- $\tau = (T-t)/365$;
- $\epsilon_{t,T}$ is small contract-specific noise.

This is not intended to be a perfect commodity model. It is a controlled synthetic environment that produces contango, backwardation, and changing curve shape.

In [ ]:
def simulate_latent_state(dates: pd.DatetimeIndex, seed: int = 42) -> pd.DataFrame:
    """
    Simulate latent spot, carry, curvature, and risk-free rate states.
    """
    local_rng = np.random.default_rng(seed)

    n = len(dates)
    dt = 1 / 252

    # Latent spot-like process.
    spot_shocks = 0.24 * np.sqrt(dt) * local_rng.standard_normal(n)
    spot_drift = 0.025 * dt
    spot = 100 * np.exp(np.cumsum(spot_drift + spot_shocks))

    time_index = np.arange(n)

    # Carry alternates between contango and backwardation regimes.
    carry = (
        0.065 * np.sin(2 * np.pi * time_index / 420)
        + 0.025 * np.sin(2 * np.pi * time_index / 115)
        + 0.008 * local_rng.standard_normal(n)
    )

    # Curvature changes more slowly.
    curvature = (
        0.035 * np.sin(2 * np.pi * time_index / 700 + 1.5)
        + 0.006 * local_rng.standard_normal(n)
    )

    # Slightly time-varying collateral rate.
    collateral_rate = (
        config.annual_collateral_rate
        + 0.005 * np.sin(2 * np.pi * time_index / 500)
        + 0.001 * local_rng.standard_normal(n)
    )
    collateral_rate = np.maximum(collateral_rate, 0.0)

    return pd.DataFrame({
        "timestamp": dates,
        "spot_index": spot,
        "carry_factor": carry,
        "curvature_factor": curvature,
        "collateral_rate": collateral_rate
    })


dates = pd.date_range(config.start, config.end, freq="B", tz="UTC")
latent_state = simulate_latent_state(dates, seed=config.seed)

latent_state.head()

In [ ]:
def simulate_contract_panel(
    contract_metadata: pd.DataFrame,
    latent_state: pd.DataFrame,
    config: FuturesCalendarConfig
) -> pd.DataFrame:
    """
    Simulate daily prices and liquidity for all live futures contracts.
    """
    local_rng = np.random.default_rng(config.seed + 1)

    frames = []

    for _, meta in contract_metadata.iterrows():
        expiry = meta["expiry"]
        contract = meta["contract"]
        delivery_month = int(meta["delivery_month"])

        part = latent_state.copy()
        part["expiry"] = expiry
        part["contract"] = contract
        part["delivery_year"] = int(meta["delivery_year"])
        part["delivery_month"] = delivery_month
        part["days_to_expiry"] = (expiry - part["timestamp"]).dt.days

        part = part[
            (part["days_to_expiry"] > 0)
            & (part["days_to_expiry"] <= config.max_days_to_expiry)
        ].copy()

        if part.empty:
            continue

        tau = part["days_to_expiry"].to_numpy() / 365.0
        spot = part["spot_index"].to_numpy()
        carry = part["carry_factor"].to_numpy()
        curvature = part["curvature_factor"].to_numpy()

        # Seasonal effect depends on delivery month.
        seasonal = 0.015 * np.sin(2 * np.pi * delivery_month / 12.0)

        curve_noise = local_rng.normal(loc=0.0, scale=0.0015, size=len(part))

        close = spot * np.exp(
            carry * tau
            + curvature * tau ** 2
            + seasonal
            + curve_noise
        )

        open_price = close * np.exp(local_rng.normal(loc=0.0, scale=0.002, size=len(part)))
        high = np.maximum(open_price, close) * (1 + np.abs(local_rng.normal(0.003, 0.002, len(part))))
        low = np.minimum(open_price, close) * (1 - np.abs(local_rng.normal(0.003, 0.002, len(part))))

        dte = part["days_to_expiry"].to_numpy()

        # Volume peaks before expiry and migrates along the curve.
        volume_shape = np.exp(-((dte - 35) ** 2) / (2 * 32 ** 2))
        open_interest_shape = np.exp(-((dte - 55) ** 2) / (2 * 50 ** 2))

        volume = 60_000 * volume_shape * local_rng.lognormal(0.0, 0.30, len(part)) + 1_000
        open_interest = 120_000 * open_interest_shape * local_rng.lognormal(0.0, 0.22, len(part)) + 2_000

        part["open"] = open_price
        part["high"] = high
        part["low"] = low
        part["close"] = close
        part["volume"] = volume
        part["open_interest"] = open_interest
        part["contract_multiplier"] = config.contract_multiplier

        frames.append(part[[
            "timestamp",
            "contract",
            "expiry",
            "delivery_year",
            "delivery_month",
            "days_to_expiry",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "open_interest",
            "contract_multiplier",
            "spot_index",
            "carry_factor",
            "curvature_factor",
            "collateral_rate"
        ]])

    contract_panel = pd.concat(frames, ignore_index=True)
    contract_panel = contract_panel.sort_values(["timestamp", "expiry"]).reset_index(drop=True)

    return contract_panel


contract_panel = simulate_contract_panel(contract_metadata, latent_state, config)

contract_panel.head()

## 6. Building a daily futures curve snapshot

For each timestamp, we rank live contracts by expiry.

The first three contracts are labelled:

- front;
- second;
- third.

These are enough to construct simple curve slope and curvature features.

In [ ]:
def build_curve_snapshots(contract_panel: pd.DataFrame) -> pd.DataFrame:
    """
    Add expiry rank to each live contract and extract front/second/third snapshots.
    """
    panel = contract_panel.sort_values(["timestamp", "expiry"]).copy()
    panel["expiry_rank"] = panel.groupby("timestamp").cumcount() + 1

    front_three = panel[panel["expiry_rank"].isin([1, 2, 3])].copy()

    wide = front_three.pivot(
        index="timestamp",
        columns="expiry_rank",
        values=["contract", "expiry", "days_to_expiry", "close", "volume", "open_interest"]
    )

    wide.columns = [
        f"{field}_{rank}"
        for field, rank in wide.columns
    ]

    wide = wide.reset_index()

    rename_map = {
        "contract_1": "front_contract",
        "contract_2": "second_contract",
        "contract_3": "third_contract",
        "expiry_1": "front_expiry",
        "expiry_2": "second_expiry",
        "expiry_3": "third_expiry",
        "days_to_expiry_1": "front_dte",
        "days_to_expiry_2": "second_dte",
        "days_to_expiry_3": "third_dte",
        "close_1": "front_price",
        "close_2": "second_price",
        "close_3": "third_price",
        "volume_1": "front_volume",
        "volume_2": "second_volume",
        "volume_3": "third_volume",
        "open_interest_1": "front_open_interest",
        "open_interest_2": "second_open_interest",
        "open_interest_3": "third_open_interest",
    }

    wide = wide.rename(columns=rename_map)

    return wide


curve_snapshots = build_curve_snapshots(contract_panel)

curve_snapshots.head()

## 7. Curve features: carry, slope, curvature

We compute three basic features.

### 7.1 Front-next carry

$$
\begin{aligned}
\text{carry}_{1,2} &= \frac{\log F_1 - \log F_2}{\tau_2-\tau_1}
\end{aligned}
$$
Positive values indicate backwardation at the front of the curve.

### 7.2 Front-third carry

$$
\begin{aligned}
\text{carry}_{1,3} &= \frac{\log F_1 - \log F_3}{\tau_3-\tau_1}
\end{aligned}
$$
This gives a longer-horizon term-structure signal.

### 7.3 Curvature

A simple log-price curvature measure is:

$$
\begin{aligned}
\text{curvature} &= \log F_1 - 2\log F_2 + \log F_3
\end{aligned}
$$
This indicates whether the first three contracts are concave or convex in log-price space.

In [ ]:
def add_curve_features(curve: pd.DataFrame) -> pd.DataFrame:
    """
    Add carry, slope, curvature, and regime labels to curve snapshots.
    """
    out = curve.copy()

    tau_1 = out["front_dte"].astype(float) / 365.0
    tau_2 = out["second_dte"].astype(float) / 365.0
    tau_3 = out["third_dte"].astype(float) / 365.0

    log_f1 = np.log(out["front_price"].astype(float))
    log_f2 = np.log(out["second_price"].astype(float))
    log_f3 = np.log(out["third_price"].astype(float))

    out["front_next_spread_pct"] = out["second_price"] / out["front_price"] - 1
    out["front_third_spread_pct"] = out["third_price"] / out["front_price"] - 1

    out["carry_1_2_annualized"] = (log_f1 - log_f2) / (tau_2 - tau_1)
    out["carry_1_3_annualized"] = (log_f1 - log_f3) / (tau_3 - tau_1)

    out["curve_curvature_log"] = log_f1 - 2 * log_f2 + log_f3

    out["term_structure_state"] = np.where(
        out["second_price"] > out["front_price"],
        "contango",
        "backwardation"
    )

    out["front_second_volume_ratio"] = out["front_volume"] / out["second_volume"]
    out["front_second_oi_ratio"] = out["front_open_interest"] / out["second_open_interest"]

    return out


curve_features = add_curve_features(curve_snapshots)

curve_features.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(curve_features["timestamp"], curve_features["carry_1_2_annualized"], label="Front-next annualised carry")
plt.axhline(0, linewidth=1)
plt.title("Front-Next Carry Signal")
plt.xlabel("Date")
plt.ylabel("Annualised log carry")
plt.legend()
plt.show()

In [ ]:
state_counts = curve_features["term_structure_state"].value_counts()

plt.figure(figsize=(7, 5))
plt.bar(state_counts.index, state_counts.values)
plt.title("Contango vs Backwardation Days")
plt.xlabel("Curve state")
plt.ylabel("Number of business days")
plt.show()

## 8. Calendar roll schedule

We now create a point-in-time roll schedule.

Rule:

> Hold the nearest contract whose expiry is more than `roll_days_before_expiry` days away.

This is a simple calendar rule. It is deterministic, auditable, and avoids using future information.

In [ ]:
def build_calendar_roll_schedule(
    contract_panel: pd.DataFrame,
    roll_days_before_expiry: int
) -> pd.DataFrame:
    """
    Build daily selected contract schedule using a calendar roll rule.
    """
    rows = []

    for timestamp, group in contract_panel.groupby("timestamp"):
        live = group[group["days_to_expiry"] > roll_days_before_expiry].sort_values("expiry")

        if live.empty:
            continue

        selected = live.iloc[0]

        rows.append({
            "timestamp": timestamp,
            "selected_contract": selected["contract"],
            "selected_expiry": selected["expiry"],
            "selected_dte": int(selected["days_to_expiry"]),
            "roll_rule": f"calendar_{roll_days_before_expiry}d"
        })

    schedule = pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)
    schedule["previous_selected_contract"] = schedule["selected_contract"].shift(1)
    schedule["is_roll_date"] = (
        schedule["selected_contract"] != schedule["previous_selected_contract"]
    ) & schedule["previous_selected_contract"].notna()

    return schedule


roll_schedule = build_calendar_roll_schedule(
    contract_panel,
    roll_days_before_expiry=config.roll_days_before_expiry
)

roll_schedule.head()

In [ ]:
roll_events = roll_schedule[roll_schedule["is_roll_date"]].copy()

roll_events.head()

## 9. Held-contract return

The economically meaningful daily futures excess return should be based on the contract actually held over the interval.

If the strategy selected contract $C_{t-1}$ at time $t-1$, then the daily held-contract return is:

$$
\begin{aligned}
r_t^{\text{held}} &= \frac{F_t(C_{t-1})}{F_{t-1}(C_{t-1})} - 1
\end{aligned}
$$
After observing $F_t(C_{t-1})$, the strategy may roll into a new contract for the next interval.

This avoids treating the roll gap itself as an immediate price return.

In [ ]:
def price_lookup_table(contract_panel: pd.DataFrame) -> pd.DataFrame:
    """
    Build a price lookup indexed by timestamp and contract.
    """
    return (
        contract_panel[["timestamp", "contract", "close", "spot_index", "collateral_rate"]]
        .copy()
        .sort_values(["timestamp", "contract"])
    )


price_lookup = price_lookup_table(contract_panel)


def compute_held_contract_returns(
    roll_schedule: pd.DataFrame,
    contract_panel: pd.DataFrame,
    config: FuturesCalendarConfig
) -> pd.DataFrame:
    """
    Compute daily held-contract excess returns and collateral returns.
    """
    schedule = roll_schedule.sort_values("timestamp").copy()
    schedule["held_contract_for_interval"] = schedule["selected_contract"].shift(1)
    schedule["previous_timestamp"] = schedule["timestamp"].shift(1)

    rows = []

    panel_indexed = contract_panel.set_index(["timestamp", "contract"]).sort_index()
    spot_by_date = contract_panel.groupby("timestamp")["spot_index"].first()
    collateral_by_date = contract_panel.groupby("timestamp")["collateral_rate"].first()

    for _, row in schedule.iloc[1:].iterrows():
        timestamp = row["timestamp"]
        previous_timestamp = row["previous_timestamp"]
        held_contract = row["held_contract_for_interval"]

        try:
            previous_price = float(panel_indexed.loc[(previous_timestamp, held_contract), "close"])
            current_price = float(panel_indexed.loc[(timestamp, held_contract), "close"])
        except KeyError:
            continue

        held_return = current_price / previous_price - 1

        previous_spot = float(spot_by_date.loc[previous_timestamp])
        current_spot = float(spot_by_date.loc[timestamp])
        spot_proxy_return = current_spot / previous_spot - 1

        annual_rate = float(collateral_by_date.loc[previous_timestamp])
        collateral_return = annual_rate / 252.0

        rows.append({
            "timestamp": timestamp,
            "previous_timestamp": previous_timestamp,
            "held_contract": held_contract,
            "selected_contract_after_close": row["selected_contract"],
            "is_roll_date": bool(row["is_roll_date"]),
            "previous_price_held_contract": previous_price,
            "current_price_held_contract": current_price,
            "held_futures_excess_return": held_return,
            "spot_proxy_return": spot_proxy_return,
            "collateral_return": collateral_return,
            "total_collateralized_return": held_return + collateral_return,
            "roll_rule": row["roll_rule"]
        })

    returns = pd.DataFrame(rows)

    return returns


held_returns = compute_held_contract_returns(
    roll_schedule,
    contract_panel,
    config
)

held_returns.head()

## 10. Roll-gap proxy return

At each roll date, compare the old and new contracts at the same timestamp:

$$
g_t = F_t^{\text{new}} - F_t^{\text{old}}
$$
For a long futures position, a common roll-gap proxy is:

$$
\begin{aligned}
\text{roll gap proxy}_t &= \frac{F_t^{\text{old}} - F_t^{\text{new}}}{F_t^{\text{old}}}
\end{aligned}
$$
This proxy is:

- negative when rolling from a lower-priced front contract into a higher-priced next contract;
- positive when rolling from a higher-priced front contract into a lower-priced next contract.

Important: this is an attribution proxy, not necessarily a literal cash PnL booked at the instant of rolling.

In [ ]:
def compute_roll_gap_proxy(
    roll_schedule: pd.DataFrame,
    contract_panel: pd.DataFrame
) -> pd.DataFrame:
    """
    Compute old/new contract prices and roll-gap proxy returns on roll dates.
    """
    events = roll_schedule[roll_schedule["is_roll_date"]].copy()

    panel_indexed = contract_panel.set_index(["timestamp", "contract"]).sort_index()

    rows = []

    for _, event in events.iterrows():
        timestamp = event["timestamp"]
        old_contract = event["previous_selected_contract"]
        new_contract = event["selected_contract"]

        try:
            old_price = float(panel_indexed.loc[(timestamp, old_contract), "close"])
            new_price = float(panel_indexed.loc[(timestamp, new_contract), "close"])
        except KeyError:
            continue

        roll_gap = new_price - old_price
        roll_gap_proxy_return = (old_price - new_price) / old_price

        rows.append({
            "timestamp": timestamp,
            "old_contract": old_contract,
            "new_contract": new_contract,
            "old_price": old_price,
            "new_price": new_price,
            "roll_gap": roll_gap,
            "roll_gap_pct_old": roll_gap / old_price,
            "roll_gap_proxy_return_long": roll_gap_proxy_return,
            "roll_state": "contango" if new_price > old_price else "backwardation"
        })

    return pd.DataFrame(rows)


roll_gap_table = compute_roll_gap_proxy(roll_schedule, contract_panel)

roll_gap_table.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(roll_gap_table["timestamp"], roll_gap_table["roll_gap_proxy_return_long"])
plt.axhline(0, linewidth=1)
plt.title("Roll-Gap Proxy Return for a Long Position")
plt.xlabel("Roll date")
plt.ylabel("Roll-gap proxy return")
plt.show()

## 11. Joining returns, curve features, and roll events

We combine:

1. held-contract returns;
2. spot proxy returns;
3. collateral return;
4. roll-gap proxy;
5. front/next curve features.

This produces a return-attribution table that can be used by downstream notebooks.

In [ ]:
attribution = held_returns.merge(
    curve_features[[
        "timestamp",
        "front_contract",
        "second_contract",
        "third_contract",
        "front_price",
        "second_price",
        "third_price",
        "front_dte",
        "second_dte",
        "third_dte",
        "front_next_spread_pct",
        "front_third_spread_pct",
        "carry_1_2_annualized",
        "carry_1_3_annualized",
        "curve_curvature_log",
        "term_structure_state",
        "front_second_volume_ratio",
        "front_second_oi_ratio"
    ]],
    on="timestamp",
    how="left"
)

attribution = attribution.merge(
    roll_gap_table[[
        "timestamp",
        "old_contract",
        "new_contract",
        "roll_gap",
        "roll_gap_pct_old",
        "roll_gap_proxy_return_long",
        "roll_state"
    ]],
    on="timestamp",
    how="left"
)

attribution["roll_gap_proxy_return_long"] = attribution["roll_gap_proxy_return_long"].fillna(0.0)

attribution["residual_vs_spot_proxy"] = (
    attribution["held_futures_excess_return"]
    - attribution["spot_proxy_return"]
)

attribution["total_return_with_roll_gap_proxy"] = (
    attribution["held_futures_excess_return"]
    + attribution["collateral_return"]
    + attribution["roll_gap_proxy_return_long"]
)

attribution.head()

## 12. Cumulative return decomposition

We compare cumulative performance of:

1. held futures excess return;
2. spot proxy return;
3. collateral return;
4. roll-gap proxy return;
5. total collateralised return.

The roll-gap proxy is shown separately because it is an attribution measure, not necessarily a directly booked daily PnL under the held-contract return convention.

In [ ]:
def cumulative_simple_return(series: pd.Series) -> pd.Series:
    """
    Convert a simple-return series to cumulative return.
    """
    return (1 + series.fillna(0.0)).cumprod() - 1


decomposition = attribution[[
    "timestamp",
    "held_futures_excess_return",
    "spot_proxy_return",
    "collateral_return",
    "roll_gap_proxy_return_long",
    "total_collateralized_return",
    "total_return_with_roll_gap_proxy"
]].copy()

for col in decomposition.columns:
    if col == "timestamp":
        continue
    decomposition[f"cum_{col}"] = cumulative_simple_return(decomposition[col])

decomposition.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(decomposition["timestamp"], decomposition["cum_held_futures_excess_return"], label="Held futures excess return")
plt.plot(decomposition["timestamp"], decomposition["cum_spot_proxy_return"], label="Spot proxy return")
plt.plot(decomposition["timestamp"], decomposition["cum_collateral_return"], label="Collateral return")
plt.plot(decomposition["timestamp"], decomposition["cum_roll_gap_proxy_return_long"], label="Roll-gap proxy")
plt.title("Cumulative Return Components")
plt.xlabel("Date")
plt.ylabel("Cumulative return")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(decomposition["timestamp"], decomposition["cum_total_collateralized_return"], label="Total collateralised return")
plt.plot(decomposition["timestamp"], decomposition["cum_held_futures_excess_return"], label="Futures excess return", alpha=0.75)
plt.plot(decomposition["timestamp"], decomposition["cum_spot_proxy_return"], label="Spot proxy return", alpha=0.75)
plt.title("Collateralised Futures Return vs Components")
plt.xlabel("Date")
plt.ylabel("Cumulative return")
plt.legend()
plt.show()

## 13. Contango/backwardation regime analysis

We now ask:

> Are futures returns and roll-gap proxies different when the front of the curve is in contango versus backwardation?

This is a basic diagnostic for carry-style strategies.

A more advanced version would test this across many commodities and use out-of-sample validation.

In [ ]:
regime_summary = (
    attribution
    .groupby("term_structure_state")
    .agg(
        n_days=("held_futures_excess_return", "count"),
        avg_held_excess_return=("held_futures_excess_return", "mean"),
        avg_spot_proxy_return=("spot_proxy_return", "mean"),
        avg_residual_vs_spot=("residual_vs_spot_proxy", "mean"),
        avg_collateral_return=("collateral_return", "mean"),
        avg_roll_gap_proxy=("roll_gap_proxy_return_long", "mean"),
        avg_carry_1_2=("carry_1_2_annualized", "mean"),
        vol_held_excess_return=("held_futures_excess_return", "std")
    )
    .reset_index()
)

regime_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(
    regime_summary["term_structure_state"],
    252 * regime_summary["avg_held_excess_return"]
)
plt.axhline(0, linewidth=1)
plt.title("Annualised Average Held Futures Excess Return by Curve State")
plt.xlabel("Term structure state")
plt.ylabel("Annualised average return")
plt.show()

## 14. Carry signal bucket analysis

We bucket days by front-next carry.

This helps test whether the carry signal has any relationship with next-day held futures returns in the synthetic data.

In a real research notebook, this would require careful out-of-sample testing, transaction costs, and multiple-testing controls.

In [ ]:
carry_analysis = attribution.copy()
carry_analysis["carry_bucket"] = pd.qcut(
    carry_analysis["carry_1_2_annualized"],
    q=5,
    labels=["lowest", "low", "middle", "high", "highest"]
)

carry_bucket_summary = (
    carry_analysis
    .groupby("carry_bucket", observed=True)
    .agg(
        n_days=("held_futures_excess_return", "count"),
        avg_next_held_excess_return=("held_futures_excess_return", "mean"),
        avg_carry=("carry_1_2_annualized", "mean"),
        vol_next_held_excess_return=("held_futures_excess_return", "std")
    )
    .reset_index()
)

carry_bucket_summary["annualized_avg_return"] = 252 * carry_bucket_summary["avg_next_held_excess_return"]

carry_bucket_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(
    carry_bucket_summary["carry_bucket"].astype(str),
    carry_bucket_summary["annualized_avg_return"]
)
plt.axhline(0, linewidth=1)
plt.title("Held Futures Return by Carry Bucket")
plt.xlabel("Carry bucket")
plt.ylabel("Annualised average held futures excess return")
plt.show()

## 15. Curve shape snapshots

A useful diagnostic is to visualise the futures curve on selected dates.

We plot several curve snapshots by days-to-expiry.

This helps check whether the simulated curves behave sensibly and whether contango/backwardation labels are intuitive.

In [ ]:
def plot_curve_snapshot(contract_panel: pd.DataFrame, timestamp: pd.Timestamp, max_rank: int = 8) -> None:
    """
    Plot a futures curve snapshot for one timestamp.
    """
    snapshot = (
        contract_panel[contract_panel["timestamp"] == timestamp]
        .sort_values("expiry")
        .head(max_rank)
        .copy()
    )

    plt.figure(figsize=(9, 5))
    plt.plot(snapshot["days_to_expiry"], snapshot["close"], marker="o")
    plt.title(f"Futures Curve Snapshot: {timestamp.date()}")
    plt.xlabel("Days to expiry")
    plt.ylabel("Futures price")
    plt.show()


snapshot_dates = [
    curve_features["timestamp"].iloc[int(len(curve_features) * q)]
    for q in [0.15, 0.35, 0.55, 0.75]
]

for snapshot_date in snapshot_dates:
    plot_curve_snapshot(contract_panel, snapshot_date, max_rank=8)

## 16. Roll calendar audit table

A roll calendar should be saved as its own object.

It should not be hidden inside a continuous price series.

This allows downstream notebooks to reproduce exactly which contract was held at every date.

In [ ]:
roll_calendar_audit = roll_schedule.merge(
    curve_features[[
        "timestamp",
        "front_contract",
        "second_contract",
        "front_price",
        "second_price",
        "carry_1_2_annualized",
        "term_structure_state",
        "front_second_volume_ratio",
        "front_second_oi_ratio"
    ]],
    on="timestamp",
    how="left"
)

roll_calendar_audit = roll_calendar_audit.merge(
    roll_gap_table,
    on="timestamp",
    how="left"
)

roll_calendar_audit.head()

## 17. Validation checks

We perform basic checks:

1. timestamps are unique in the roll schedule;
2. selected contracts exist on each timestamp;
3. held-contract return intervals are well-defined;
4. curve features are finite when front/second/third contracts exist;
5. roll gaps are available on roll dates where both contracts are live.

These checks are not exhaustive, but they make assumptions explicit.

In [ ]:
def validate_roll_decomposition(
    roll_schedule: pd.DataFrame,
    attribution: pd.DataFrame,
    curve_features: pd.DataFrame,
    roll_gap_table: pd.DataFrame
) -> pd.DataFrame:
    """
    Validate basic properties of the futures calendar and decomposition.
    """
    rows = []

    rows.append({
        "check": "unique_roll_schedule_timestamps",
        "passed": roll_schedule["timestamp"].is_unique,
        "detail": int(roll_schedule["timestamp"].duplicated().sum())
    })

    rows.append({
        "check": "no_missing_selected_contract",
        "passed": roll_schedule["selected_contract"].notna().all(),
        "detail": int(roll_schedule["selected_contract"].isna().sum())
    })

    rows.append({
        "check": "held_returns_finite",
        "passed": np.isfinite(attribution["held_futures_excess_return"]).all(),
        "detail": int((~np.isfinite(attribution["held_futures_excess_return"])).sum())
    })

    feature_cols = [
        "carry_1_2_annualized",
        "carry_1_3_annualized",
        "curve_curvature_log"
    ]

    for col in feature_cols:
        finite_mask = np.isfinite(curve_features[col])
        rows.append({
            "check": f"{col}_finite",
            "passed": bool(finite_mask.all()),
            "detail": int((~finite_mask).sum())
        })

    roll_count = int(roll_schedule["is_roll_date"].sum())
    gap_count = len(roll_gap_table)

    rows.append({
        "check": "roll_gap_coverage",
        "passed": gap_count == roll_count,
        "detail": f"{gap_count} gap rows for {roll_count} roll dates"
    })

    return pd.DataFrame(rows)


validation_checks = validate_roll_decomposition(
    roll_schedule=roll_schedule,
    attribution=attribution,
    curve_features=curve_features,
    roll_gap_table=roll_gap_table
)

validation_checks

## 18. Saving datasets

The notebook saves three auditable datasets:

1. `curve_features.csv`  
   Daily front/second/third curve features.

2. `roll_calendar.csv`  
   Daily selected contract and roll events.

3. `return_attribution.csv`  
   Held-contract returns, spot proxy returns, collateral returns, and roll-gap proxy returns.

It also saves a JSON manifest with the construction assumptions.

In [ ]:
output_dir = Path("data/processed/futures_roll_yield_decomposition_v1")
output_dir.mkdir(parents=True, exist_ok=True)

curve_features_path = output_dir / "curve_features.csv"
roll_calendar_path = output_dir / "roll_calendar.csv"
attribution_path = output_dir / "return_attribution.csv"
roll_gap_path = output_dir / "roll_gap_table.csv"
validation_path = output_dir / "validation_checks.csv"
manifest_path = output_dir / "manifest.json"

curve_features.to_csv(curve_features_path, index=False)
roll_calendar_audit.to_csv(roll_calendar_path, index=False)
attribution.to_csv(attribution_path, index=False)
roll_gap_table.to_csv(roll_gap_path, index=False)
validation_checks.to_csv(validation_path, index=False)

manifest = {
    "dataset_name": "synthetic_futures_roll_yield_decomposition",
    "schema_version": "futures_roll_decomposition_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "n_contract_rows": int(len(contract_panel)),
    "n_curve_feature_rows": int(len(curve_features)),
    "n_roll_calendar_rows": int(len(roll_calendar_audit)),
    "n_attribution_rows": int(len(attribution)),
    "n_roll_events": int(len(roll_gap_table)),
    "notes": [
        "Synthetic data only.",
        "Roll-gap proxy is an attribution measure, not literal cash PnL.",
        "Held-contract return is based on contract held over each interval.",
        "Collateral return is approximated using a daily risk-free accrual."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

curve_features_path, roll_calendar_path, attribution_path, manifest_path

## 19. Practical interpretation checklist

Before using futures roll-yield features in alpha research, ask:

1. **What is the roll rule?**  
   Calendar, volume, open interest, first notice day, liquidity threshold, or strategy-specific?

2. **What is the return convention?**  
   Held-contract return, adjusted continuous return, excess return, total return, or index return?

3. **Is roll yield a realised PnL or attribution proxy?**  
   The distinction matters.

4. **Is collateral return included?**  
   Fully collateralised futures returns include cash collateral return.

5. **Is the curve state point-in-time?**  
   Do not use future liquidity or revised metadata to define today’s roll.

6. **Does the signal use adjusted prices?**  
   Back-adjusted chart prices may not be appropriate for absolute carry calculations.

7. **Are transaction costs included?**  
   Rolling is not free.

8. **Are contract multipliers and margin rules included?**  
   Required for realistic PnL and risk.

9. **Is the product seasonal?**  
   Many commodities have delivery-month seasonality.

10. **Does the strategy trade the curve or only the front contract?**  
   Calendar-spread and outright strategies have different exposures.

## 20. Limitations

This notebook deliberately uses a simplified synthetic futures market.

### 20.1 Spot is not directly observable for many futures markets

The notebook uses a latent spot proxy.

In real commodity markets, spot prices may be fragmented, location-specific, storage-dependent, or unavailable at the same quality/grade as the futures contract.

### 20.2 Roll yield definitions vary

Different sources define roll yield differently.

Some compare spot and futures returns. Some use futures curve slope. Some compute index roll returns during a roll window. Some treat roll yield as an accounting residual.

A robust research pipeline should define the convention explicitly.

### 20.3 Synthetic carry dynamics are simplified

The simulated carry and curvature factors create plausible curve shapes, but they do not model inventories, storage costs, convenience yield, seasonality, funding, or delivery constraints in detail.

### 20.4 Roll execution is simplified

The notebook does not model:

- bid-ask spread;
- market impact;
- partial fills;
- roll window execution;
- crowding;
- limit moves;
- exchange holidays;
- night sessions;
- margin changes.

### 20.5 Single-asset results do not prove an alpha

Carry effects should be tested across many markets, with transaction costs, out-of-sample validation, and multiple-testing controls.

### 20.6 Adjusted price series are not enough

A back-adjusted continuous price may be useful for trend signals, but roll yield and curve carry should usually be calculated from actual contract prices at the same timestamp.

## 21. Research frontier and current directions

Futures term-structure research is broader than simple front-next roll yield.

### 21.1 Curve-aware features instead of one continuous contract

A single continuous futures series throws away most of the curve.

Modern research often treats the curve as a vector or function across maturities.

A future notebook could use PCA or dynamic factor models to extract level, slope, and curvature factors from the futures curve.

### 21.2 Carry, momentum, and volatility as separate signals

Commodity futures research often studies whether term structure, momentum, and volatility contain distinct information.

A future notebook could build a triple-screen signal using:

- carry or roll-yield proxy;
- time-series momentum;
- idiosyncratic volatility.

### 21.3 State-space models for futures curves

Futures curves can be modelled with latent factors such as short-term shocks, long-term equilibrium, slope, curvature, and stochastic volatility.

A future notebook could implement a Kalman filter for a two-factor commodity futures model.

### 21.4 Signature methods for term structures

Recent mathematical finance research applies path signatures to futures term structures.

The idea is to treat the evolving curve as a path and extract features that capture its shape and dynamics.

A future notebook could compare simple carry/slope features with signature features on synthetic futures curves.

### 21.5 Execution-aware rolling

Institutional rolling is not just a calendar decision.

It depends on:

- liquidity;
- open interest;
- volume;
- spread;
- market impact;
- crowding;
- exchange rules;
- delivery risk.

A future notebook could simulate roll execution over several days and attribute roll cost separately from curve carry.

### 21.6 Exchange-specific futures calendars

For Chinese commodity futures, the calendar layer needs product-specific metadata:

- night sessions;
- margin schedule changes;
- daily limit rules;
- delivery-month restrictions;
- contract code conventions;
- exchange holiday calendars;
- dominant-contract definitions.

A future notebook could extend the schema to DCE, CZCE, SHFE, and INE-style metadata.

## 22. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_10_statistical_arbitrage_pairs**  
   Using calendar spreads and curve relationships in stat-arb research.

2. **03_14_information_coefficient_and_feature_decay**  
   Testing whether carry features predict future returns.

3. **04_08_futures_minimum_variance_hedge_ratio**  
   Using futures contracts for hedge construction.

4. **05_01_vectorized_backtest_engine**  
   Backtesting futures strategies using held-contract returns and roll schedules.

5. **05_03_transaction_costs_slippage_latency**  
   Adding realistic costs to roll execution.

6. **06_07_limit_board_margin_and_deadband_rebalancing**  
   Adding exchange-specific futures constraints.

7. **07_01_multi_asset_cta_research_pipeline**  
   Integrating futures calendar, carry, momentum, risk, and backtesting.

## 23. Summary

This notebook built a futures calendar and roll-yield decomposition layer.

The main outputs were:

1. a synthetic futures contract panel;
2. daily front/second/third curve snapshots;
3. carry, spread, and curvature features;
4. a calendar-based roll schedule;
5. held-contract futures excess returns;
6. spot proxy returns;
7. collateral returns;
8. roll-gap proxy returns;
9. contango/backwardation regime diagnostics;
10. carry bucket analysis;
11. saved curve, calendar, attribution, and manifest files.

The key computational takeaway is:

> Futures data should be represented as contracts, calendars, curves, and returns — not only as one smoothed continuous price column.

The key financial takeaway is:

> Roll yield is not a magic extra return. It is a curve-dependent attribution concept that must be defined carefully and separated from spot movement, collateral return, and actual held-contract PnL.

## 24. Further reading

### Futures return decomposition

- CME Group material on deconstructing futures returns and roll yield.
- CME Group material on contango and backwardation.
- Commodity futures literature on spot return, roll yield, and collateral return.
- Practitioner discussions of futures excess return versus total return.

### Commodity futures term structure

- Fuertes, Miffre, and Rallis on momentum, term structure, and idiosyncratic volatility in commodity futures.
- Research on cost-of-carry, storage, convenience yield, and inventory effects.
- Studies of curve factors such as level, slope, and curvature.

### Current research directions

- State-space modelling of commodity futures curves.
- Functional regression models for futures term structures.
- Path signature methods for futures curve dynamics.
- Execution-aware roll scheduling.
- Point-in-time futures metadata and calendar construction.